<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/14_Deep_Learning/02_CNN_Satellite_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 14: Deep Learning Foundations
## Part 02: Convolutional Neural Networks (CNN) for Spatial Satellite Vision

### The Spatial Challenge: Why We Must Move Beyond MLPs
In the previous phase, our Multilayer Perceptron (MLP) successfully identified anomalies in 1D tabular sensor streams. However, MLPs suffer from a fatal flaw when handling visual or spatial data: they require flat inputs. If we feed a 2D satellite image into an MLP, we must flatten the pixel grid into a single row. This process completely destroys spatial hierarchy—the MLP loses the ability to recognize that a dark green pixel is adjacent to a brown pixel, erasing critical geographic boundaries like deforestation lines or urban sprawl.

To process spatial data, we require **translation invariance** (the ability to recognize a feature regardless of where it appears in the image) and **localized feature extraction**. Convolutional Neural Networks (CNNs) solve this by sliding mathematically localized filters (kernels) across the 2D grid, preserving the exact geometry and texture of the original terrain.

### Objective
Transition from 1-dimensional point observations to 2-dimensional spatial grids. We will construct a Convolutional Neural Network to classify localized land-cover anomalies (e.g., rapid deforestation or severe drought stress) using multispectral satellite image tensors, ensuring the final architecture remains strictly optimized for edge-device mobile deployment.

### Key Milestones
1. **Spatial Tensor Generation:** Ingest and synthesize 2D RGB/Multispectral satellite image tiles representing varying land-cover conditions.
2. **Tensor Preprocessing:** Normalize the spatial pixel grids to stable float32 scales (0.0 to 1.0) for stable gradient descent.
3. **Hierarchical Architecture:** Construct a CNN using `Conv2D` layers to extract geometric features and `MaxPooling2D` to downsample the spatial dimensions, significantly reducing the parameter count for mobile processing.
4. **Classification & Validation:** Flatten the extracted spatial maps into a dense classification head, compile the network, and evaluate its precision on a blind spatial holdout set.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Create image data subdirectories
os.makedirs("data/satellite_tiles", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Verify environment
print(f"TensorFlow Version: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"⚡ GPU Detected: {gpu_devices}")
else:
    print("ℹ️ Running on CPU backend.")

# Reproducibility seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TensorFlow Version: 2.21.0
ℹ️ Running on CPU backend.
